# Search

Run after `python -m similarity.preprocessing`.

In [1]:
# Work from the repo root, so `similarity` imports and the data/ paths resolve when opened from notebooks/
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [2]:
import time
import numpy as np
import pandas as pd
import similarity.search as s
from similarity.preprocessing import load_products

products = load_products()
f = s.load_features()
s.WEIGHTS

{'text': 0.3,
 'category': 0.2,
 'gender': 0.1,
 'price': 0.15,
 'brand': 0.15,
 'colour': 0.05,
 'weight': 0.05}

## one product

In [3]:
row = 3
products.loc[row, ["name", "brand", "child_category", "price", "colours", "weight_g"]]

name              People Men's Printed Regular fit T-Shirt
brand                                                 None
child_category                                MensT_Shirts
price                                                195.0
colours                                               None
weight_g                                               NaN
Name: 3, dtype: object

In [4]:
order, score = s.rank(f, row)
products.loc[order[:10], ["name", "child_category", "price", "rating_adjusted"]].assign(score=score[:10].round(3))

,name,child_category,price,rating_adjusted,score
3277,People Men's Printed Regular fit T-Shirt,MensT_Shirts,199.0,4.175367,0.996
24907,People Men's Printed Regular fit T-Shirt,MensT_Shirts,179.0,3.812806,0.984
24596,People Men's Regular fit T-Shirt,MensT_Shirts,199.0,4.229473,0.967
20922,People Men's Regular fit T-Shirt,MensT_Shirts,199.0,4.175367,0.967
9410,People Men's Regular fit T-Shirt,MensT_Shirts,199.0,4.146139,0.967
26769,People Men's Regular fit T-Shirt,MensT_Shirts,199.0,3.982405,0.967
3117,People Men's Regular fit T-Shirt,MensT_Shirts,179.0,4.146139,0.954
17189,People Men's Regular fit T-Shirt,MensT_Shirts,179.0,3.868119,0.954
6642,People Men's Regular fit T-Shirt,MensT_Shirts,219.0,4.261052,0.949
13815,People Men's Regular fit T-Shirt,MensT_Shirts,219.0,3.839548,0.949


In [5]:
s.find_similar_products(products.loc[row, "uniq_id"], 5)

['0e5673848b57d123bb464cd6dc9bc0d1',
 'c09569e8b45404f7c26a9a431746b9db',
 'd46f4ce37e6385af94e00f95895b93f7',
 'e6e1f01084865da2f272b9fb54d4dcf3',
 '17a1e2f69dce7616cccc670509f254e4']

## breakdown

In [6]:
s.score_breakdown(f, row, order[0]).round(3)

,weight,present,score,rescaled_weight,contribution
text,0.30,True,1.00,0.400,0.400
category,0.20,True,1.00,0.267,0.267
gender,0.10,True,1.00,0.133,0.133
price,0.15,True,0.98,0.200,0.196
brand,0.15,False,NaN,0.000,0.000
colour,0.05,False,NaN,0.000,0.000
weight,0.05,False,NaN,0.000,0.000


In [7]:
s.score_breakdown(f, row, 1).round(3)

,weight,present,score,rescaled_weight,contribution
text,0.30,True,0.149,0.400,0.060
category,0.20,True,1.000,0.267,0.267
gender,0.10,True,1.000,0.133,0.133
price,0.15,True,0.736,0.200,0.147
brand,0.15,False,NaN,0.000,0.000
colour,0.05,False,NaN,0.000,0.000
weight,0.05,False,NaN,0.000,0.000


In [8]:
scores = s.attribute_scores(f, row)
rows = [1, 7, 2]
pd.DataFrame({
    "name": products.loc[rows, "name"].str[:45],
    "text_only": scores["text"][0][rows].round(3),
    "combined": s.combine(scores, s.WEIGHTS)[rows].round(3),
})

,name,text_only,combined
1,Sf Jeans By Pantaloons Men's Plain Slim fit T,0.149,0.607
7,Pepe Jeans Men's Solid Regular fit Casual Shi,0.220,0.264
2,LOVISTA Cotton Gota Patti Tassel Traditional,0.057,0.082


## missing attributes

In [9]:
has = pd.DataFrame({
    "text": products["name"].str.len() > 0,
    "category": (f.category != -1) | (f.browsenode != -1),
    "gender": f.gender != -1,
    "price": ~np.isnan(f.price),
    "brand": f.brand != -1,
    "colour": f.colours.getnnz(axis=1) > 0,
    "weight": ~np.isnan(f.weight),
})
has.mean().round(3)

text        1.000
category    0.997
gender      0.912
price       0.904
brand       0.720
colour      0.201
weight      0.206
dtype: float64

In [10]:
has.sum(axis=1).value_counts().sort_index()

2       16
3      869
4     8772
5    12554
6     6842
7      947
Name: count, dtype: int64

In [11]:
few = has.index[has.sum(axis=1) == 2][0]
has.loc[few]

text         True
category     True
gender      False
price       False
brand       False
colour      False
weight      False
Name: 491, dtype: bool

In [12]:
order, score = s.rank(f, few)
pd.concat([
    products.loc[[few], ["name", "child_category", "price"]],
    products.loc[order[:5], ["name", "child_category", "price"]].assign(score=score[:5].round(3)),
])

,name,child_category,price,score
491,Marvel Avengers Hydra Logo Cuff Beanie Black,None,NaN,NaN
12344,Pokemon Pokeball Cuff Pom Beanie Winter Hat,None,3155.0,0.612
24055,Cuffed Beanie - Warm Daily Beanie Hat with Fol...,None,NaN,0.610
15377,E4hats White Diamond Embroidered Long Cuff Bea...,None,NaN,0.600
25743,K-9 Silver Logo Embroidery Embroidered Beanie ...,None,NaN,0.593
17973,Knit Woodland Camo Cuff Long Beanie - Winter W...,None,NaN,0.577


In [13]:
s.score_breakdown(f, few, order[0]).round(3)

,weight,present,score,rescaled_weight,contribution
text,0.30,True,0.353,0.6,0.212
category,0.20,True,1.000,0.4,0.400
gender,0.10,False,NaN,0.0,0.000
price,0.15,False,NaN,0.0,0.000
brand,0.15,False,NaN,0.0,0.000
colour,0.05,False,NaN,0.0,0.000
weight,0.05,False,NaN,0.0,0.000


## duplicates

In [14]:
asin = products["asin"].value_counts().index[0]
twins = products.index[products["asin"] == asin]
products.loc[twins, ["uniq_id", "asin", "name", "price"]]

,uniq_id,asin,name,price
184,981b995e1e8016ef7c17b98b782aa88f,B07MMDR5CP,Captain America By Kidsville Boy's Plain Regul...,395.0
9690,6f44f9f4080f41049b748d6d52837eaa,B07MMDR5CP,Captain America By Kidsville Boy's Plain Regul...,269.0
18236,696e7e5b79d6727ec816ade2431f41df,B07MMDR5CP,Captain America By Kidsville Boy's Plain Regul...,395.0


In [15]:
order, score = s.rank(f, twins[0])
products.loc[order[:5], ["uniq_id", "asin", "name"]].assign(score=score[:5].round(3))

,uniq_id,asin,name,score
12064,45347d99ea5a17411ec227291f2bd2dd,B07SPZMM94,Captain America By Kidsville Boy's Regular fit...,0.974
26628,90d573b15625ae85cc7047579711a97e,B07M7WW58G,Captain America By Kidsville Boy's Plain Regul...,0.958
10921,dd8c3640df8f189da0d9de85de6cad1a,B07MMDQLHZ,Captain America By Kidsville Boy's Plain Regul...,0.945
21295,c2a3eba0e17e0ff42095943daa436c94,B07P9R4P1X,FLYING MACHINE Boy's Plain Regular fit T-Shirt,0.734
996,e1b571c06e92da8f484a7888ef6329ed,B07L1GZ11F,612 League Boy's Plain Regular fit T-Shirt,0.723


In [16]:
(products.loc[order, "asin"] == asin).sum(), products.loc[order, "asin"].is_unique, len(order)

(np.int64(0), True, 29528)

## ties

In [17]:
order, score = s.rank(f, row)
top = products.loc[order[:20], ["name", "rating_adjusted"]].assign(score=score[:20].round(3))
top[top["score"].duplicated(keep=False)]

,name,rating_adjusted,score
24596,People Men's Regular fit T-Shirt,4.229473,0.967
20922,People Men's Regular fit T-Shirt,4.175367,0.967
9410,People Men's Regular fit T-Shirt,4.146139,0.967
26769,People Men's Regular fit T-Shirt,3.982405,0.967
3117,People Men's Regular fit T-Shirt,4.146139,0.954
17189,People Men's Regular fit T-Shirt,3.868119,0.954
6642,People Men's Regular fit T-Shirt,4.261052,0.949
13815,People Men's Regular fit T-Shirt,3.839548,0.949
8870,People Men's Regular fit T-Shirt,3.830760,0.949
2970,People Men's Regular fit T-Shirt,4.372105,0.945


## random products

In [18]:
r = products.sample(1, random_state=1).index[0]
order, score = s.rank(f, r)
pd.concat([
    products.loc[[r], ["name", "child_category", "price"]],
    products.loc[order[:5], ["name", "child_category", "price"]].assign(score=score[:5].round(3)),
])

,name,child_category,price,score
10747,PalmTree Boys' Sweatshirt,BoysSweatshirtsHoodies,599.0,NaN
10227,Jockey Boys' Sweatshirt,BoysSweatshirtsHoodies,700.0,0.754
27663,Pepe Jeans Boys' Sweatshirt,BoysSweatshirtsHoodies,616.0,0.751
18475,Allen Solly Boys Sweatshirt,BoysSweatshirtsHoodies,616.0,0.738
19158,Pepe Jeans Boys' Sweatshirt,BoysSweatshirtsHoodies,542.0,0.738
22269,Allen Solly Boys Sweatshirt,BoysSweatshirtsHoodies,616.0,0.738


In [19]:
r = products.sample(1, random_state=2).index[0]
order, score = s.rank(f, r)
pd.concat([
    products.loc[[r], ["name", "child_category", "price"]],
    products.loc[order[:5], ["name", "child_category", "price"]].assign(score=score[:5].round(3)),
])

,name,child_category,price,score
7945,Everking Kids Soft Cotton Drawer/Briefs Innerw...,None,395.0,NaN
22496,Everking Kids Soft Cotton Panties/Briefs Inner...,GirlsKnickers,385.0,0.935
6869,Everking Kids Soft Cotton Panties/Briefs Inner...,GirlsKnickers,419.0,0.929
29962,Everking Kids Soft Cotton Panties/Briefs Inner...,GirlsKnickers,399.0,0.904
5829,My Baby Boy's and Girl's Drawer/Panties (Multi...,None,399.0,0.671
29438,Kids Basket Cartoon Baby Boys and Girls Cotton...,None,399.0,0.600


## no shared words

In [20]:
order, score = s.rank(f, 0)
text = s.text_scores(f, 0)[0]
products.loc[order[:8], ["name", "child_category", "price"]].assign(text=text[order[:8]].round(2), score=score[:8].round(3))

,name,child_category,price,text,score
10141,Cotton Kalamkari Handblock Saree Blouse/Kurti ...,WomensEthnicUnstitchedFabric,200.0,0.6,0.645
8211,Rangmanch by Pantaloons Women's Polyester Blou...,WomensKurtasKurtis,200.0,0.0,0.600
5118,Indigo Women's Straight Kurti,WomensKurtasKurtis,201.0,0.0,0.599
5748,Indigo Women's Straight Kurti,WomensKurtasKurtis,198.0,0.0,0.598
11230,Rangriti Women's Straight Kurta,WomensKurtasKurtis,203.0,0.0,0.597
27510,Amazon Brand - Myx Women's A-Line Kurti,WomensKurtasKurtis,206.0,0.0,0.594
14959,Amazon Brand - Myx Women's Straight Kurti,WomensKurtasKurtis,194.0,0.0,0.594
18165,Amazon Brand - Myx Women's Rayon Straight Kurti,WomensKurtasKurtis,193.0,0.0,0.593


In [21]:
closest = np.argsort(-text)[1:6]
position = pd.Series(np.arange(len(order)), index=order)
products.loc[closest, ["name", "child_category", "price"]].assign(text=text[closest].round(2), rank=position.reindex(closest).values)

,name,child_category,price,text,rank
18915,LA' Facon Cotton Kalamkari Handblock Saree Blo...,WomensKurtasKurtis,200.0,1.00,NaN
10141,Cotton Kalamkari Handblock Saree Blouse/Kurti ...,WomensEthnicUnstitchedFabric,200.0,0.60,0.0
315,Looks Akira Cotton with Blouse Piece Saree (LA...,WomensSarees,399.0,0.25,3768.0
19612,HANDBLOCK PRINT Women's Rayon Embroidered Anar...,WomensKurtasKurtis,543.0,0.24,540.0
5360,Phoenix Retail's Kalamkari Maysore Silk Saree ...,WomensSarees,449.0,0.24,3552.0


## text only vs combined

In [22]:
rows = np.random.default_rng(0).choice(len(f.ids), 500, replace=False)
text_only = s.result_checks(f, rows, {"text": 1.0})
combined = s.result_checks(f, rows)
pd.DataFrame({"text_only": text_only.mean(), "combined": combined.mean()}).round(3)

,text_only,combined
same_category,0.678,0.976
same_gender,0.876,0.997
price_ratio,0.731,0.870
no_shared_words,0.000,0.019
evidence_top,0.823,0.791
evidence_all,0.786,0.786


## timing

In [23]:
start = time.perf_counter()
for r in range(100):
    s.rank(f, r)
(time.perf_counter() - start) / 100 * 1000

23.99187199996959